# Estimating Faraday Rotation from NISAR GSLC Data 

<br>

A Faraday Rotation estimate allows us to estimate how much the ionosphere has rotated the polarization of the SAR signal. 


Faraday rotation in the ionosphere scales with the square of a SAR signal's wavelength ($\lambda^2$). Therefore, ionospheric impacts due to polarization rotation are much more pronounced in NISAR's ~25cm L-band data than in Sentinel-1's ~5cm C-band data.

This notebook loads NISAR L-band GSLC data and performs a Faraday rotation estimation using the circular-basis method described in equation 14 of [F. J. Meyer and J. B. Nicoll's, “Prediction, detection, and correction of Faraday rotation in full-polarimetric
    L-band SAR data”](https://topex.ucsd.edu/pub/sandwell/Venus_InSAR/atmosphere/meyer_nicoll_TGRS46_10_ionosphere_Polarimetry.pdf).

<hr>

## Overview

1. [Prerequisites](faraday-prereqs)
1. [Search for data](faraday-search)
1. [Load a single GSLC with HTTPS](faraday-https)
1. [Define functions to perform a Faraday rotation estimation over an optionally multilooked GSLC](faraday-functions)
1. [Estimate Faraday rotation on the multi-looked subset GSLC](faraday-estimate)
1. [Plot the Faraday rotation over a basemap](faraday-plot)
1. [Summary](faraday-summary)
1. [Resources and references](faraday-resources)

<hr>

(faraday-prereqs)=
## 1. Prerequisites
| Prerequisite | Importance | Notes |
| --- | --- | --- |
| [The software environment for this cookbook must be installed](https://github.com/ASFOpenSARlab/NISAR_GCOV_Cookbook/blob/main/notebooks/create_software_environment.ipynb) | Necessary | |

- **Rough Notebook Time Estimate**: 10 minutes

<hr>

(faraday-search)=
## 2. Search for data

:::{warning}
GSLC data must be acquired in quad-pol mode (containing all four linear polarizations) to perform a Faraday rotation estimation using the circular-basis method. 
:::

Quad-pol data are only acquired in some areas, so it may be helpful to use Vertex to visually identify suitable AOIs prior to searching programmatically.

:::{figure} ../assets/vertex_quad_pol.png
:alt: Image of a quad-pol GSLC search on Vertex
:width: 75%
:align: left

Quad-pol GSLC search on Vertex

[Example quad-pol GSLC search on Vertex](https://search.asf.alaska.edu/#/?sciProducts=GSLC&polarizations=HH-HV-VH-VV&resultsLoaded=true&granule=NISAR_L2_PR_GSLC_033_034_D_072_2005_QPDH_A_20261014T133841_20261014T133916_P05023_N_F_J_001). 

Select "Quad Pol" from the L-Band Polarization filter.
:::

### 2a. Perform an `asf_search.search()`

Adjust the search parameters below to find a quad-pol GSLC product.


In [1]:
import asf_search as asf
from datetime import datetime
from getpass import getpass

import warnings
warnings.filterwarnings(
    "ignore",
    message="Parsing dates involving a day of month without a year specified",
)

session = asf.ASFSession()

start_date = datetime(2026, 7, 26)
end_date = datetime(2026, 12, 31)
area_of_interest = "POINT(447.0504 24.5906)" # POINT or POLYGON as WKT (well-known-text)
pattern = r'^(?=.*QP)(?!.*QA_STATS).*' # Select QP (quad-pol) data and disregard QA_STATS products

opts=asf.ASFSearchOptions(**{
    "maxResults": 250,
    "intersectsWith": area_of_interest,
    "flightDirection": "DESCENDING",
    "start": start_date,
    "end": end_date,
    "dataMaturity" : "PROVISIONAL",
    "processingLevel": ["GSLC"],
    "dataset": ["NISAR"],
    "productionConfiguration": ["PR"],
    "session": session,
})

response = asf.search(opts=opts)
hdf5_urls = response.find_urls(extension='.h5', pattern=pattern, directAccess=False)
print(f"Found {len(hdf5_urls)} GSLC products:")
hdf5_urls

["'type': 'WRAP': 'report': 'Wrapped 1 value(s) to +/-180 longitude'"]


Found 4 GSLC products:


['https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GSLC_PROVISIONAL_V1/NISAR_L2_PR_GSLC_028_120_D_076_2005_QPDH_A_20260821T125100_20260821T125134_P05023_N_F_J_001/NISAR_L2_PR_GSLC_028_120_D_076_2005_QPDH_A_20260821T125100_20260821T125134_P05023_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GSLC_PROVISIONAL_V1/NISAR_L2_PR_GSLC_029_120_D_076_2005_QPDH_A_20260902T125059_20260902T125134_P05023_N_F_J_001/NISAR_L2_PR_GSLC_029_120_D_076_2005_QPDH_A_20260902T125059_20260902T125134_P05023_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GSLC_PROVISIONAL_V1/NISAR_L2_PR_GSLC_030_120_D_076_2005_QPDH_A_20260914T125059_20260914T125134_P05023_N_F_J_001/NISAR_L2_PR_GSLC_030_120_D_076_2005_QPDH_A_20260914T125059_20260914T125134_P05023_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GSLC_PROVISIONAL_V1/NISAR_L2_PR_GSLC_031_120_D_076_2005_QPDH_A_20260926T125059_20260926T125134_P05023_N_F_J_001/NISAR_L2_PR_GSLC_031_120_

### 2b. Select a url from the search results

In [2]:
hdf5_url = hdf5_urls[0]

hdf5_url

'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GSLC_PROVISIONAL_V1/NISAR_L2_PR_GSLC_028_120_D_076_2005_QPDH_A_20260821T125100_20260821T125134_P05023_N_F_J_001/NISAR_L2_PR_GSLC_028_120_D_076_2005_QPDH_A_20260821T125100_20260821T125134_P05023_N_F_J_001.h5'

### 2c. Provide your Earthdata Login (EDL) Bearer Token

Both HTTPS and S3 access require an EDL Bearer Token

:::{figure} ../assets/edl_token.png
:alt: Image of the Earthdata Login Generate Token web page
:width: 75%
:align: left

View or generate a Bearer Token in "Generate Token" tab of the Profile page in your Earthdata Login account: https://urs.earthdata.nasa.gov/profile
:::

:::{note} Temporary S3 credentials

The S3 credentials acquired below expire after 1 hour.
:::

In [6]:
token = getpass("Enter your EDL Bearer Token")

<hr>

(faraday-https)=
## 3. Stream and subset a GSLC with HTTP

### 3a. Lazily load a GSLC dataset with `xarray`

In [7]:
%%time
import asf_search as asf
import fsspec
import rioxarray
import xarray as xr

group_path = "/science/LSAR/GSLC/grids/frequencyA"

fs = fsspec.filesystem(
    "http",
    headers = {"Authorization": f"Bearer {token}"},
    block_size = 16 * 512 * 512,
)

f = fs.open(hdf5_url, "rb")

dt = xr.open_datatree(
    f,
    engine="h5netcdf",
    decode_timedelta=False,
    phony_dims="access",
    chunks="auto",
    group=group_path
)

display(dt)

<xarray.DataTree>
Group: /
    Dimensions:                 (yCoordinates: 67680, xCoordinates: 34488,
                                 phony_dim_0: 4)
    Coordinates:
      * yCoordinates            (yCoordinates) float64 541kB 2.907e+06 ... 2.569e+06
      * xCoordinates            (xCoordinates) float64 276kB 3.442e+05 ... 6.89e+05
    Dimensions without coordinates: phony_dim_0
    Data variables: (12/16)
        HH                      (yCoordinates, xCoordinates) complex64 19GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
        HV                      (yCoordinates, xCoordinates) complex64 19GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
        VH                      (yCoordinates, xCoordinates) complex64 19GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
        VV                      (yCoordinates, xCoordinates) complex64 19GB dask.array<chunksize=(4096, 4096), meta=np.ndarray>
        azimuthBandwidth        float64 8B ...
        centerFrequency         float64 8B ...
        ...                      ...
        projection              uint32 4B ...
        rangeBandwidth          float64 8B ...
        slantRangeSpacing       float64 8B ...
        xCoordinateSpacing      float64 8B ...
        yCoordinateSpacing      float64 8B ...
        zeroDopplerTimeSpacing  float64 8B ...

CPU times: user 969 ms, sys: 281 ms, total: 1.25 s
Wall time: 16.3 s


### 3b. Create a spatial subset 

In [8]:
import rioxarray

ds = dt.to_dataset()
projection = ds.projection.attrs['epsg_code'].item()
ds = ds.rio.write_crs(projection) # write the project to the HHHH data for easy lat/lon subsetting

# subset the data
subset_ds = ds.rio.clip_box(
    minx=446.5857, miny=24.7935,
    maxx=446.7342, maxy=24.9244, 
    crs="EPSG:4326"
)
subset_ds

<xarray.Dataset> Size: 162MB
Dimensions:                 (xCoordinates: 1505, yCoordinates: 2908,
                             phony_dim_0: 4)
Coordinates:
  * xCoordinates            (xCoordinates) float64 12kB 4.581e+05 ... 4.732e+05
  * yCoordinates            (yCoordinates) float64 23kB 2.757e+06 ... 2.742e+06
    projection              int64 8B 0
Dimensions without coordinates: phony_dim_0
Data variables: (12/15)
    HH                      (yCoordinates, xCoordinates) complex64 35MB dask.array<chunksize=(2625, 892), meta=np.ndarray>
    HV                      (yCoordinates, xCoordinates) complex64 35MB dask.array<chunksize=(2625, 892), meta=np.ndarray>
    VH                      (yCoordinates, xCoordinates) complex64 35MB dask.array<chunksize=(2625, 892), meta=np.ndarray>
    VV                      (yCoordinates, xCoordinates) complex64 35MB dask.array<chunksize=(2625, 892), meta=np.ndarray>
    azimuthBandwidth        float64 8B ...
    centerFrequency         float64 8B ...
    ...                      ...
    numberOfSubSwaths       uint8 1B ...
    rangeBandwidth          float64 8B ...
    slantRangeSpacing       float64 8B ...
    xCoordinateSpacing      float64 8B ...
    yCoordinateSpacing      float64 8B ...
    zeroDopplerTimeSpacing  float64 8B ...

<hr>

(faraday-functions)=
## 4. Perform a Faraday Rotation estimation and calculate a quality metric

### 4a. Define functions to perform a Faraday Rotation estimation over an optionally multilooked GSLC

TODO: Add LaTex for equations used below

Uses the circular-basis method for Faraday rotation estimation described in equation 14 of [F. J. Meyer and J. B. Nicoll's, “Prediction, detection, and correction of Faraday rotation in full-polarimetric
    L-band SAR data”](https://topex.ucsd.edu/pub/sandwell/Venus_InSAR/atmosphere/meyer_nicoll_TGRS46_10_ionosphere_Polarimetry.pdf)

In [10]:
import numpy as np
import xarray as xr

def multilook_mean(da, ly=1, lx=1, x_coord="xCoordinates", y_coord="yCoordinates"):
    """
    Spatially multilook an `xarray.DataArray` by taking the mean over
    pixel windows (ly x lx).

    da : (xarray.Dataset or xarray.DataArray) 2D DataArray to multilook
    ly : (int) Number of pixels to average together in the y direction
    lx : (int) Number of pixels to average together in the x direction

    returns: Multilooked xarray.DataArray
    """
    return da.coarsen(
        {y_coord: ly, x_coord: lx},
        boundary="trim"
    ).mean()


def calculate_circular_terms(hh, hv, vh, vv):
    """
        Z = T * M' * T
    
        T = [[1, j],
             [j, 1]]
    
    [[Z11 Z12], =  [[1, j],      [[hh, vh],    [[1, j],
    [Z21 Z22]]  =  [j, 1]]   *   [hv, vv]]  *  [j, 1]]
    """
    
    T = 1j
    # calculate the circular-basis off-diagonal terms 
    z12 = (T * hh) - vh + hv + (T * vv)
    z21 = (T * hh) + vh - hv + (T * vv)

    return z12, z21

def get_phase_product(z12, z21, ly=1, lx=1, x_coord="xCoordinates", y_coord="yCoordinates"):
    # Find cross-product of off-diagonal terms
    phase_product = z12 * np.conj(z21)

    # multilook the cross-product of off-diagonal terms
    return multilook_mean(
        phase_product,
        ly=ly, lx=lx,
        x_coord=x_coord, y_coord=y_coord
    )

def faraday_rotation_circular_basis(phase_product_ml, ly=1, lx=1, x_coord="xCoordinates", y_coord="yCoordinates"):
    """
    Estimate Faraday rotation angle using the circular-basis method:

    [Z11 Z12]   [1  j] [M'hh M'vh] [1  j]
    [Z21 Z22] = [j  1] [M'hv M'vv] [j  1]

    from which we derive:

    Ω_hat = 1/4 * arg(Z12 * conj(Z21))

    M′: measured scattering matrices (linear basis)
    Ω_hat: Faraday Rotation angle estimate
    Z: circular basis (Z12: RL, Z21: LR)
    arg: argument of complex number (phase in the complex plane)

    References: 
    - F. J. Meyer and J. B. Nicoll, “Prediction, detection, and correction of Faraday rotation in full-polarimetric
    L-band SAR data,” IEEE Trans. Geosci. Remote Sens., vol. 46, no. 10, pp. 3076–3086, Oct. 2008.
    https://topex.ucsd.edu/pub/sandwell/Venus_InSAR/atmosphere/meyer_nicoll_TGRS46_10_ionosphere_Polarimetry.pdf
    - S. H. Bickel and R. H. T. Bates, “Effects of magneto-ionic propagation on the
    polarization scattering matrix,” Proc. IEEE, vol. 53, no. 8, pp. 1089–1091, Aug. 1965

    hh, hv, vh, vv : (xarray.Dataset or xarray.DataArray) Complex-valued, linearly polarized image DataArrays
    ly : (int) Number of looks in y for multilooking
    lx : (int) Number of looks in x for multilooking
    x_coord: (string) DataArrays' x coordinate variable name
    y_coord: (sring) DataArrays' y coordinate variable name

    Returns: omega (xarray.DataArray) Estimated Faraday rotation angle in radians
    """


    # P = Z12 * conj(Z21)
    # Ω_hat = 1/4 * arg(P)
    # and
    # Z=x+iy
    # cosθ = x/hyp, sinθ = y/hyp
    # θ = arctan(y/x)
    # ∴
    # arg(P)=atan2(ℑ(P),ℜ(P))
    # Ω_hat = 1/4 * atan2(ℑ(P),ℜ(P))
    omega_rad = 0.25 * xr.apply_ufunc(
        np.arctan2, # np.arctan2 avoids quadrant ambiguity
        phase_product_ml.imag, # y-coord
        phase_product_ml.real, # x-coord
        dask="parallelized",
        output_dtypes=[np.float64],
    )
    omega_rad.name = "faraday_rotation_rad"
    omega_rad.attrs["units"] = "radian"

    return omega_rad


def coherence_based_quality(z12, z21, phase_product_ml, ly=1, lx=1, x_coord="xCoordinates", y_coord="yCoordinates"):
    """
    Provides the local complex (backscatter amplitude and phase) correlation magnitude between the two circular-basis channels
    
    Coherence-based quality metric
    P = Z12 * conj(Z21)
    quality = |P| / sqrt(|Z12|^2 * |Z21|^2)
    
    quality ≈ 1: stable phase difference, reliable FR estimate
    quality ≈ 0: random/noisy phase difference, unreliable FR estimate
    """
    z12_pow_ml = multilook_mean(
        np.abs(z12) ** 2,
        ly=ly, lx=lx,
        x_coord=x_coord, y_coord=y_coord
    )

    z21_pow_ml = multilook_mean(
        np.abs(z21) ** 2,
        ly=ly, lx=lx,
        x_coord=x_coord, y_coord=y_coord
    )

    quality = np.abs(phase_product_ml) / np.sqrt(z12_pow_ml * z21_pow_ml)
    quality = quality.clip(0, 1)
    quality.name = "faraday_rotation_quality"
    quality.attrs["units"] = "unitless"

    return quality

### 4b. Calculate the circular basis, Faraday Rotation estimate, and quality metric

:::{hint}
The GSLC data is lazily loaded, so this step will run quickly. However, the upcoming plot step may run very slowly because that is when all the data will be loaded into memory and queued operations performed. 
:::

In [11]:
z12, z21 = calculate_circular_terms(    
    subset_ds.HH,
    subset_ds.HV,
    subset_ds.VH,
    subset_ds.VV
)

lx, ly = 10, 10 # multilooking window

phase_product_ml = get_phase_product(z12, z21, ly=ly, lx=lx)
omega_rad = faraday_rotation_circular_basis(phase_product_ml, ly=ly, lx=lx)
quality = coherence_based_quality(z12, z21, phase_product_ml, ly=ly, lx=lx)

if hasattr(subset_ds.HH, "rio") and subset_ds.HH.rio.crs is not None:
    omega_rad = omega_rad.rio.write_crs(subset_ds.HH.rio.crs, inplace=False)
    quality = quality.rio.write_crs(subset_ds.HH.rio.crs, inplace=False)

<hr>

(faraday-plot)=
## 6. Plot the Faraday rotation over a basemap

:::{warning} The cell below may run slowly on large datasets

Lazily-loaded data are materialized in memory and queued calculations are performed during this step. 
:::


>"Faraday rotation (FR) angles of less than $5^\circ$ are generally considered acceptable for many commonly used SAR parameter extraction methods. Higher levels of FR can introduce significant errors in SAR image interpretation and analysis, particularly for polarimetric decompositions that rely on channel ratios."
>
>
>*[Meyer, F. J., and J. B. Nicoll (2008), “Prediction, detection, and correction of Faraday rotation in full-polarimetric L-band SAR data,” IEEE Transactions on Geoscience and Remote Sensing, 46(10), 3076–3086](https://topex.ucsd.edu/pub/sandwell/Venus_InSAR/atmosphere/meyer_nicoll_TGRS46_10_ionosphere_Polarimetry.pdf)*

In [ ]:
%%time

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import panel as pn
import holoviews as hv
import hvplot.xarray  # noqa
from util.plot import fr_view

hv.extension("bokeh")
pn.extension()

omega_deg = np.degrees(omega_rad)
omega_deg.name = "faraday_rotation_deg"
omega_deg.attrs["units"] = "degree"

quality_thresh = 0.3 # Set quality threshold for masking 
omega_deg_masked = omega_deg.where(quality >= quality_thresh)
omega_deg_masked.name = "faraday_rotation_deg_quality_masked"
omega_deg_masked.attrs["units"] = "degree"
omega_deg_masked.attrs["quality_threshold"] = quality_thresh
omega_deg_masked

omega_web = omega_deg_masked.rio.reproject("EPSG:3857")
quality_web = quality.rio.reproject_match(omega_web)
omega_web = omega_web.compute()
quality_web = quality_web.compute()

print(f"FR min/max: {float(omega_web.min())}, {float(omega_web.max())}")
mean_abs_deg = float(np.abs(omega_deg_masked).mean(skipna=True))
print(f"Mean abs FR: {mean_abs_deg}")
median_abs_deg = float(np.nanmedian(np.abs(omega_deg_masked).compute()))
print(f"Median abs FR: {median_abs_deg}")
std_deg = float(omega_deg_masked.std(skipna=True))
print(f"Std FR: {std_deg}")
print(f"Quality min/max: {float(quality_web.min())}, {float(quality_web.max())}")
print(f"Quality valid pixels: {int(quality_web.count())}")

xmin = float(omega_web.x.min())
xmax = float(omega_web.x.max())
ymin = float(omega_web.y.min())
ymax = float(omega_web.y.max())

alpha = pn.widgets.FloatSlider(name="FR alpha", start=0.0, end=1.0, step=0.05, value=1.0)
show_fr = pn.widgets.Toggle(name="Show Faraday rotation", value=True)
show_quality = pn.widgets.Toggle(name="Show quality", value=False)

tiles = hv.element.tiles.EsriImagery().opts(
    xlim=(xmin, xmax),
    ylim=(ymin, ymax),
    frame_width=700,
    frame_height=500,
    active_tools=["wheel_zoom"],
)

pn.Column(show_fr, alpha, show_quality)

<timed exec>:12: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.


FR min/max: -11.224892616271973, 28.43159294128418
Mean abs FR: 4.966257495157931
Median abs FR: 4.9632568359375
Std FR: 2.3794150349292855
Quality min/max: 0.1773519217967987, 0.9962980151176453
Quality valid pixels: 53357
CPU times: user 46.5 s, sys: 13.4 s, total: 59.9 s
Wall time: 9min 33s


ModuleNotFoundError: No module named 'jupyter_bokeh'

Column
    [0] Toggle(label='Show Faraday rotation', name='Show Faraday rotation', value=True)
    [1] FloatSlider(label='FR alpha', name='FR alpha', step=0.05, value=1.0)
    [2] Toggle(label='Show quality', name='Show quality')

In [19]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

plot_data = {
    "show_fr": omega_web,
    "show_quality": quality_web,
    "meta": {
        "bounds": {
            "x": (xmin, xmax),
            "y": (ymin, ymax)
        }
    }
}

fr_view(plot_data=plot_data, tiles=tiles, alpha=alpha)

:Overlay
   .Tiles.I  :Tiles   [x,y]
   .Image.I  :Image   [x,y]   (faraday_rotation_deg_quality_masked)
   .Image.II :Image   [x,y]   (faraday_rotation_quality)

<hr>

(faraday-summary)=
## 7. Summary
You now have the tools and knowledge that you need to search GSLC data with asf_search, stream it with ffspec and xarray, and estimate Faraday rotation caused by passage of the signal through the ionosphere.   

<hr>

(faraday-resources)=
## 8. Resources and references
 - [asf_search](https://github.com/asfadmin/Discovery-asf_search)
 - [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/aws-s3-access/)
 - [F. J. Meyer and J. B. Nicoll's, “Prediction, detection, and correction of Faraday rotation in full-polarimetric
    L-band SAR data”](https://topex.ucsd.edu/pub/sandwell/Venus_InSAR/atmosphere/meyer_nicoll_TGRS46_10_ionosphere_Polarimetry.pdf)
 
**Author:** Alex Lewandowski